# New Hire Onboarding Packet

Event-driven onboarding automation: when HR adds an employee to a SharePoint
"New Hires" list, the sandbox automatically provisions resources and sends notifications.

```
SharePoint List (new item) → Gateway Trigger → Sandbox (Flask handler)
                                                  ├── Create OneDrive folder /Onboarding/{Name}
                                                  ├── Send welcome email to new hire
                                                  └── Send heads-up email to manager
```

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- SDKs: `pip install azure-connectorgateway azure-sandbox azure-mgmt-sandbox`
- A SharePoint site with a "New Hires" list

Click **Run All** or go **Step by Step**.

### 0. Initialize

In [ ]:
import os, sys, json, subprocess, time
from azure.connectorgateway import ConnectorGatewayClient, TriggerClient
from azure.sandbox import SandboxClient
from azure.mgmt.sandbox import SandboxGroupManagementClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

_SHELL = sys.platform == 'win32'
account = json.loads(subprocess.run(
    ['az', 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True, shell=_SHELL).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'eastus2'
sandbox_group_name = f'{lab_name}-sg'
gateway_name = f'{lab_name}-gw'
sharepoint_connection_name = 'sharepoint-conn'
office365_connection_name = 'o365-conn'
trigger_config_name = 'new-hire-trigger'

# SharePoint site and list configuration
sharepoint_site_url = os.environ.get('SHAREPOINT_SITE_URL', 'https://your-tenant.sharepoint.com/sites/HR')
sharepoint_list_name = os.environ.get('SHAREPOINT_LIST_NAME', 'New Hires')

trigger_client = TriggerClient(subscription_id=subscription_id, resource_group=resource_group_name)
conn_client = ConnectorGatewayClient(subscription_id=subscription_id, resource_group=resource_group_name)
sbx_client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupManagementClient(subscription_id=subscription_id, resource_group=resource_group_name)

print(f'Lab:              {lab_name}')
print(f'User:             {account["user"]["name"]}')
print(f'Subscription:     {account["name"]} ({subscription_id})')
print(f'Resource Group:   {resource_group_name}')
print(f'Gateway:          {gateway_name}')
print(f'Sandbox Group:    {sandbox_group_name}')
print(f'SharePoint Site:  {sharepoint_site_url}')
print(f'SharePoint List:  {sharepoint_list_name}')

### 1. Create resource group and connector gateway

The gateway needs a SystemAssigned managed identity to authenticate callbacks to the sandbox.

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

try:
    gw = conn_client.get_gateway(gateway_name)
    print(f'Gateway: {gateway_name} (exists)')
except Exception:
    gw = conn_client.create_gateway(gateway_name, location='brazilsouth',
        identity={'type': 'SystemAssigned'})
    print(f'Gateway: {gateway_name} (created)')

gw_principal_id = gw['identity']['principalId']
gw_tenant_id = gw['identity']['tenantId']
print(f'Principal ID: {gw_principal_id}')

### 2. Create OAuth connections

We need two connections:
- **SharePoint** — trigger source (detects new list items)
- **Office 365** — send emails and manage OneDrive folders

If not yet authorized, click the consent links immediately — they expire quickly.

In [ ]:
def ensure_connection(name, connector):
    """Create connection if needed and check auth status."""
    try:
        conn_client.create_connection(gateway_name, name, connector_name=connector)
        print(f'Connection: {name} (created)')
    except Exception as e:
        if '409' in str(e) or 'Conflict' in str(e):
            print(f'Connection: {name} (already exists)')
        else:
            raise

    conn = conn_client.get_connection(gateway_name, name)
    status = conn.get('properties', {}).get('statuses', [{}])[0].get('status', 'Unknown')
    print(f'  Status: {status}')

    if status != 'Connected':
        link = conn_client.generate_consent_link(gateway_name, name)
        print(f'  ⚠️  Authorize here (click immediately):\n  {link}')
    else:
        print(f'  ✅ Already connected!')
    return conn

print('--- SharePoint Connection ---')
sp_conn = ensure_connection(sharepoint_connection_name, 'sharepointonline')
print()
print('--- Office 365 Connection ---')
o365_conn = ensure_connection(office365_connection_name, 'office365')

### 3. Create sandbox

The sandbox hosts our onboarding handler — a Flask app that receives the
SharePoint trigger payload and orchestrates the onboarding steps.

In [ ]:
group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Sandbox group: {group["name"]}')

sandboxes = sbx_client.list_sandboxes(sandbox_group_name)
if sandboxes:
    sandbox_id = sandboxes[0]['id']
    print(f'Sandbox: {sandbox_id} (existing)')
else:
    # Retry with backoff — data plane needs time to register the new group
    for attempt in range(6):
        try:
            sbx = sbx_client.create_sandbox(sandbox_group_name, disk='ubuntu')
            sandbox_id = sbx['id']
            print(f'Sandbox: {sandbox_id} (created)')
            break
        except Exception as e:
            if attempt < 5 and 'SandboxGroupNotFound' in str(e):
                wait = (attempt + 1) * 10
                print(f'  Waiting {wait}s for sandbox group to propagate...')
                time.sleep(wait)
            else:
                raise

print('Waiting for sandbox...')
resumed = False
for i in range(36):
    info = sbx_client.get_sandbox(sandbox_id, sandbox_group_name)
    state = info.get('state', 'Unknown')
    print(f'  [{i+1}] state={state}')
    if state == 'Running':
        break
    if state == 'Idle' and not resumed:
        sbx_client.resume_sandbox(sandbox_id, sandbox_group_name)
        resumed = True
    time.sleep(5)
else:
    raise RuntimeError('Sandbox did not reach Running state in 3 minutes')

print(f'\u2705 Sandbox is running!')

### 4. Install dependencies

Install Flask and requests in the sandbox.

In [ ]:
r = sbx_client.exec(sandbox_id, sandbox_group_name,
    'dpkg -s python3-flask >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq python3-flask python3-requests 2>&1 | tail -1)')
print(f'Dependencies: {r["stdout"].strip() or "already installed"}')

### 5. Deploy the Onboarding Handler

A Flask app that:
- **POST /webhook** — receives SharePoint trigger payloads (new hire list items)
- Parses new hire data (name, role, start date, manager email, employee email)
- Creates a OneDrive folder `/Onboarding/{Name}` with a welcome document
- Sends a welcome email to the new hire
- Sends a heads-up email to the manager
- **GET /** — dashboard showing onboarding activity

In [ ]:
app_code = """\
from flask import Flask, request, jsonify
import json, datetime, os, re, html as html_lib

app = Flask(__name__)
onboarding_log = []

# Configuration — set via environment or defaults
GRAPH_BASE = 'https://graph.microsoft.com/v1.0'


def extract_new_hire(payload):
    \"\"\"Extract new hire fields from the SharePoint trigger payload.\"\"\"
    body = payload.get('body', payload)
    if isinstance(body, dict) and 'value' in body:
        items = body['value']
    elif isinstance(body, dict):
        items = [body]
    else:
        items = [body] if body else []

    results = []
    for item in items:
        fields = item.get('fields', item)
        results.append({
            'name': fields.get('Title', fields.get('name', 'Unknown')),
            'role': fields.get('Role', fields.get('role', 'Team Member')),
            'start_date': fields.get('StartDate', fields.get('startDate', 'TBD')),
            'manager_email': fields.get('ManagerEmail', fields.get('managerEmail', '')),
            'employee_email': fields.get('EmployeeEmail', fields.get('employeeEmail', '')),
        })
    return results


def create_onedrive_folder(name):
    \"\"\"Create /Onboarding/{Name} folder in OneDrive via Graph API.\"\"\"
    import requests
    token = os.environ.get('GRAPH_TOKEN', '')
    if not token:
        return {'status': 'skipped', 'reason': 'no GRAPH_TOKEN configured'}

    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

    # Ensure /Onboarding folder exists
    requests.post(f'{GRAPH_BASE}/me/drive/root/children',
        headers=headers,
        json={'name': 'Onboarding', 'folder': {}, '@microsoft.graph.conflictBehavior': 'fail'})

    # Create employee subfolder
    safe_name = re.sub(r'[^\\w\\s-]', '', name).strip()
    resp = requests.post(f'{GRAPH_BASE}/me/drive/root:/Onboarding/{safe_name}:/children',
        headers=headers,
        json={'name': 'Welcome.md', 'file': {},
              '@microsoft.graph.sourceUrl': 'data:text/plain,Welcome to the team!'})

    # Upload a welcome document
    welcome_content = f\"\"\"# Welcome to the Team, {name}!

We're excited to have you join us. Here's what to expect on your first day:

1. Badge pickup at the front desk (Building A, Floor 1)
2. Laptop setup with IT (Room 204)
3. Team lunch at 12:00 PM
4. Meet your manager for a 1:1 at 2:00 PM

## Useful Links
- [Company Handbook](https://intranet.example.com/handbook)
- [Benefits Portal](https://intranet.example.com/benefits)
- [IT Help Desk](https://intranet.example.com/it)

See you soon!
\"\"\"
    requests.put(
        f'{GRAPH_BASE}/me/drive/root:/Onboarding/{safe_name}/Welcome.md:/content',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'text/plain'},
        data=welcome_content.encode())

    return {'status': 'created', 'path': f'/Onboarding/{safe_name}'}


def send_welcome_email(employee_email, name, role, start_date):
    \"\"\"Send welcome email to the new hire via Graph API.\"\"\"
    import requests
    token = os.environ.get('GRAPH_TOKEN', '')
    if not token:
        return {'status': 'skipped', 'reason': 'no GRAPH_TOKEN configured'}
    if not employee_email:
        return {'status': 'skipped', 'reason': 'no employee email'}

    body = {
        'message': {
            'subject': f'Welcome aboard, {name}! 🎉',
            'body': {
                'contentType': 'HTML',
                'content': f\"\"\"<h2>Welcome to the team, {name}!</h2>
<p>We're thrilled to have you joining us as <b>{role}</b> starting <b>{start_date}</b>.</p>
<h3>First-Day Instructions</h3>
<ul>
  <li>🏢 Report to the front desk at <b>9:00 AM</b> for badge pickup</li>
  <li>💻 IT will have your laptop ready in Room 204</li>
  <li>🍽️ Team lunch at 12:00 PM — we'll come get you!</li>
  <li>📋 Your onboarding folder is ready: <a href='https://onedrive.example.com/Onboarding/{name}'>OneDrive/Onboarding/{name}</a></li>
</ul>
<h3>Useful Links</h3>
<ul>
  <li><a href='https://intranet.example.com/handbook'>Company Handbook</a></li>
  <li><a href='https://intranet.example.com/benefits'>Benefits Portal</a></li>
  <li><a href='https://intranet.example.com/it'>IT Help Desk</a></li>
</ul>
<p>See you on {start_date}! 🚀</p>\"\"\"
            },
            'toRecipients': [{'emailAddress': {'address': employee_email}}]
        },
        'saveToSentItems': 'true'
    }

    resp = requests.post(f'{GRAPH_BASE}/me/sendMail',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
        json=body)
    return {'status': 'sent' if resp.status_code == 202 else 'error',
            'code': resp.status_code}


def send_manager_email(manager_email, name, role, start_date):
    \"\"\"Send heads-up email to the manager via Graph API.\"\"\"
    import requests
    token = os.environ.get('GRAPH_TOKEN', '')
    if not token:
        return {'status': 'skipped', 'reason': 'no GRAPH_TOKEN configured'}
    if not manager_email:
        return {'status': 'skipped', 'reason': 'no manager email'}

    body = {
        'message': {
            'subject': f'Heads up: Your new report {name} starts {start_date}',
            'body': {
                'contentType': 'HTML',
                'content': f\"\"\"<h2>New Team Member Starting Soon</h2>
<p>Hi! Just a heads up that your new report is joining the team:</p>
<table style='border-collapse:collapse;margin:16px 0'>
  <tr><td style='padding:8px;font-weight:bold'>Name:</td><td style='padding:8px'>{name}</td></tr>
  <tr><td style='padding:8px;font-weight:bold'>Role:</td><td style='padding:8px'>{role}</td></tr>
  <tr><td style='padding:8px;font-weight:bold'>Start Date:</td><td style='padding:8px'>{start_date}</td></tr>
</table>
<h3>What's been set up:</h3>
<ul>
  <li>✅ OneDrive onboarding folder created</li>
  <li>✅ Welcome email sent to {name}</li>
  <li>📋 Please schedule a 1:1 for their first afternoon</li>
</ul>
<p>Their onboarding docs are at: <a href='https://onedrive.example.com/Onboarding/{name}'>OneDrive/Onboarding/{name}</a></p>\"\"\"
            },
            'toRecipients': [{'emailAddress': {'address': manager_email}}]
        },
        'saveToSentItems': 'true'
    }

    resp = requests.post(f'{GRAPH_BASE}/me/sendMail',
        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'},
        json=body)
    return {'status': 'sent' if resp.status_code == 202 else 'error',
            'code': resp.status_code}


@app.route('/webhook', methods=['POST'])
def webhook():
    \"\"\"Handle SharePoint trigger — new hire added to list.\"\"\"
    payload = request.get_json(silent=True) or {}
    new_hires = extract_new_hire(payload)

    results = []
    for hire in new_hires:
        entry = {
            'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'hire': hire,
            'actions': {}
        }

        # Step 1: Create OneDrive folder
        entry['actions']['folder'] = create_onedrive_folder(hire['name'])

        # Step 2: Send welcome email to new hire
        entry['actions']['welcome_email'] = send_welcome_email(
            hire['employee_email'], hire['name'], hire['role'], hire['start_date'])

        # Step 3: Send manager notification
        entry['actions']['manager_email'] = send_manager_email(
            hire['manager_email'], hire['name'], hire['role'], hire['start_date'])

        onboarding_log.append(entry)
        results.append(entry)

    return jsonify(status='processed', count=len(results), results=results)


@app.route('/')
def dashboard():
    \"\"\"Onboarding activity dashboard.\"\"\"
    html = '<html><head><style>'
    html += 'body{font-family:sans-serif;max-width:900px;margin:0 auto;padding:20px}'
    html += '.card{border:1px solid #ddd;border-radius:8px;padding:16px;margin:12px 0;border-left:5px solid #0078d4}'
    html += '.stats{display:flex;gap:20px;margin:16px 0} .stat{padding:12px 24px;border-radius:8px;text-align:center}'
    html += '.stat-total{background:#e7f3ff;color:#0078d4} .stat-ok{background:#d4edda;color:#155724} .stat-skip{background:#fff3cd;color:#856404}'
    html += 'table{border-collapse:collapse;width:100%} td,th{padding:8px;border-bottom:1px solid #eee;text-align:left}'
    html += '</style></head><body>'
    html += '<h1>&#128188; New Hire Onboarding Dashboard</h1>'
    html += '<p>Automated onboarding activity from SharePoint trigger</p>'

    total = len(onboarding_log)
    ok = sum(1 for e in onboarding_log
             if e['actions'].get('welcome_email', {}).get('status') == 'sent')
    skipped = total - ok

    html += '<div class="stats">'
    html += f'<div class="stat stat-total"><b>{total}</b><br>Total Hires</div>'
    html += f'<div class="stat stat-ok"><b>{ok}</b><br>Fully Processed</div>'
    html += f'<div class="stat stat-skip"><b>{skipped}</b><br>Partial/Skipped</div>'
    html += '</div><hr>'

    if not onboarding_log:
        html += '<p><i>No onboarding events yet. Add a new hire to your SharePoint list to trigger the flow!</i></p>'
    else:
        for entry in reversed(onboarding_log[-20:]):
            h = entry['hire']
            html += '<div class="card">'
            html += f'<h3>&#128100; {h["name"]}</h3>'
            html += f'<table>'
            html += f'<tr><td><b>Role</b></td><td>{h["role"]}</td></tr>'
            html += f'<tr><td><b>Start Date</b></td><td>{h["start_date"]}</td></tr>'
            html += f'<tr><td><b>Email</b></td><td>{h["employee_email"]}</td></tr>'
            html += f'<tr><td><b>Manager</b></td><td>{h["manager_email"]}</td></tr>'
            html += '</table>'
            html += '<p><b>Actions:</b></p><ul>'
            for action, result in entry['actions'].items():
                icon = '&#9989;' if result.get('status') in ('sent', 'created') else '&#9888;&#65039;'
                html += f'<li>{icon} {action}: {result.get("status", "unknown")}</li>'
            html += '</ul>'
            html += f'<small>{entry["timestamp"]}</small>'
            html += '</div>'

    html += '</body></html>'
    return html


@app.route('/api/log')
def api_log():
    return jsonify(onboarding_log=onboarding_log, total=len(onboarding_log))


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

sbx_client.exec(sandbox_id, sandbox_group_name, 'pkill -f server.py 2>/dev/null; sleep 1; true')
sbx_client.write_file(sandbox_id, sandbox_group_name, '/app/server.py', app_code.strip())
print('Uploaded /app/server.py')

sbx_client.exec(sandbox_id, sandbox_group_name, 'nohup python3 /app/server.py > /tmp/server.log 2>&1 &')
time.sleep(2)
r = sbx_client.exec(sandbox_id, sandbox_group_name, 'curl -s http://localhost:5000/ | head -c 100')
print(f'Server: {"running" if "Onboarding" in r["stdout"] else "starting..."}')

try:
    sbx_client.add_port(sandbox_id, sandbox_group_name, 5000, anonymous=True)
except Exception:
    pass

dashboard_url = f'https://{sandbox_id}--5000.proxy.azuredevcompute.io'
print(f'\nDashboard: {dashboard_url}')

### 6. Discover SharePoint trigger operations

Lists available trigger operations from the SharePoint Online connector.

In [ ]:
try:
    ops = trigger_client.list_trigger_operations(gateway_name, 'sharepointonline')
    print(f'Found {len(ops)} trigger operations for sharepointonline:\n')
    for op in ops:
        print(f'  • {op["operationId"]}: {op.get("summary", "")}')
        print(f'    Type: {op.get("triggerType", "unknown")}')
except Exception as e:
    print(f'Could not discover operations: {e}')
    print('\nUsing known operation: OnNewListItem (Batch trigger - polls for new items)')

### 7. Create trigger config + access policy

Creates a SharePoint trigger that fires when a new item is added to the
"New Hires" list. The trigger POSTs the new item data to our sandbox webhook.

Safe to re-run — deletes and recreates if exists.

In [ ]:
try:
    trigger_client.delete_trigger(gateway_name, trigger_config_name)
    print(f'Deleted existing trigger config: {trigger_config_name}')
except Exception:
    pass

trigger = trigger_client.create_trigger(
    gateway_name, trigger_config_name,
    connector_name='sharepointonline',
    connection_name=sharepoint_connection_name,
    operation_name='OnNewListItem',
    sandbox_id=sandbox_id,
    sandbox_group=sandbox_group_name,
    port=5000,
    port_path='/webhook',
    http_method='POST',
    description='New Hire Onboarding - triggers when HR adds employee to SharePoint list',
    parameters=[
        {'name': 'siteUrl', 'value': sharepoint_site_url},
        {'name': 'listName', 'value': sharepoint_list_name},
    ],
)
print(f'✅ Trigger config created: {trigger_config_name}')
print(f'State: {trigger["properties"]["state"]}')
print(f'Callback: {trigger["properties"]["notificationDetails"]["callbackUrl"]}')

# Grant access policy for SharePoint connection
try:
    conn_client.create_access_policy(
        gateway_name, sharepoint_connection_name,
        principal_id=gw_principal_id,
        tenant_id=gw_tenant_id,
        location='brazilsouth')
    print(f'✅ Access policy granted for SharePoint connection')
except Exception as e:
    if '409' in str(e) or 'Conflict' in str(e):
        print('✅ SharePoint access policy already exists')
    else:
        raise

# Grant access policy for Office 365 connection
try:
    conn_client.create_access_policy(
        gateway_name, office365_connection_name,
        principal_id=gw_principal_id,
        tenant_id=gw_tenant_id,
        location='brazilsouth')
    print(f'✅ Access policy granted for Office 365 connection')
except Exception as e:
    if '409' in str(e) or 'Conflict' in str(e):
        print('✅ Office 365 access policy already exists')
    else:
        raise

### 8. Verify trigger is active

In [ ]:
tc = trigger_client.get_trigger(gateway_name, trigger_config_name)
state = tc['properties']['state']
print(f'Trigger config state: {state}')
if state == 'Enabled':
    print('✅ Trigger is active and listening for new SharePoint list items!')
else:
    print(f'⚠️  State is {state} — may take a moment to activate')

### 9. Test it — add a new hire to the SharePoint list

Go to your SharePoint site's "New Hires" list and add a new item with:

| Column | Example Value |
|--------|---------------|
| Title | Jane Smith |
| Role | Software Engineer |
| StartDate | 2025-06-02 |
| ManagerEmail | manager@company.com |
| EmployeeEmail | jane.smith@company.com |

Wait 30–60 seconds for the trigger to poll and fire, then run the cell below.

Alternatively, simulate a trigger with a direct POST:

In [ ]:
# Simulate a new hire trigger payload for testing
test_payload = json.dumps({
    'body': {
        'value': [{
            'fields': {
                'Title': 'Jane Smith',
                'Role': 'Software Engineer',
                'StartDate': '2025-06-02',
                'ManagerEmail': 'manager@company.com',
                'EmployeeEmail': 'jane.smith@company.com'
            }
        }]
    }
})

r = sbx_client.exec(sandbox_id, sandbox_group_name,
    f"curl -s -X POST http://localhost:5000/webhook -H 'Content-Type: application/json' -d '{test_payload}'")
result = json.loads(r['stdout'])
print(json.dumps(result, indent=2))

### 10. Check the dashboard

In [ ]:
r = sbx_client.exec(sandbox_id, sandbox_group_name, 'curl -s http://localhost:5000/api/log')
data = json.loads(r['stdout'])
print(f'Onboarding events processed: {data["total"]}')
for entry in data['onboarding_log'][-5:]:
    h = entry['hire']
    actions = entry['actions']
    folder_status = actions.get('folder', {}).get('status', '?')
    welcome_status = actions.get('welcome_email', {}).get('status', '?')
    manager_status = actions.get('manager_email', {}).get('status', '?')
    print(f'  👤 {h["name"]} ({h["role"]}) — start: {h["start_date"]}')
    print(f'     📁 Folder: {folder_status} | 📧 Welcome: {welcome_status} | 📧 Manager: {manager_status}')

print(f'\nDashboard: {dashboard_url}')

### 11. Lifecycle management

In [ ]:
result = trigger_client.disable_trigger(gateway_name, trigger_config_name)
print(f'⏸️  Disabled: state={result["properties"]["state"]}')
time.sleep(2)

result = trigger_client.enable_trigger(gateway_name, trigger_config_name)
print(f'▶️  Enabled: state={result["properties"]["state"]}')

print('\n--- Full trigger config ---')
detail = trigger_client.get_trigger(gateway_name, trigger_config_name)
print(json.dumps(detail, indent=2, default=str))

### 12. Clean up

In [ ]:
trigger_client.delete_trigger(gateway_name, trigger_config_name)
print(f'Deleted trigger: {trigger_config_name}')

sbx_client.delete_sandbox(sandbox_id, sandbox_group_name)
print(f'Deleted sandbox')

mgmt.delete_group(sandbox_group_name)
print(f'Deleted sandbox group')

conn_client.delete_gateway(gateway_name)
print(f'Deleted gateway')

!az group delete --name {resource_group_name} --yes --no-wait
print(f'Deleting resource group (async)')